# PushT ACT (Official Repo) - Training Notebook

Built on the official ACT policy code with a thin adapter for the PushT replay dataset.

## Notebook Roadmap
- Setup and imports
- Dataset resolution and normalization
- Dataset wrapper and dataloaders
- Policy build and optimizer setup
- Training loop with rollout-based selection
- Final evaluation and video export

## 1. Setup and Imports
Basic setup, paths, and core imports used across the notebook.

In [ ]:
import csv
import os
import random
import sys
from collections import deque
from pathlib import Path

import gymnasium as gym
import numpy as np
import torch
import zarr
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Paths
PROJECT_ROOT = Path.cwd().resolve()
OFFICIAL_ROOT = PROJECT_ROOT / 'official_act_repo_probe'
OFFICIAL_DETR_ROOT = OFFICIAL_ROOT / 'detr'
ACT_ENV_ROOT = PROJECT_ROOT / 'ACT_pusht_task' / 'env'
DATA_CANDIDATES = [
    PROJECT_ROOT / 'ACT_pusht_task' / 'data',
    PROJECT_ROOT / 'ACT_pusht_task' / 'data' / 'pusht_cchi_v7_replay.zarr',
    PROJECT_ROOT / 'ACT_pusht_task' / 'data' / 'pusht_cchi_v7_replay.zarr.zip',
]

if not OFFICIAL_ROOT.is_dir():
    raise FileNotFoundError(f'official_act_repo_probe not found at: {OFFICIAL_ROOT}')

# Official ACT imports need both repo root and `detr`; simulator rollout needs env path.
for _p in (OFFICIAL_ROOT, OFFICIAL_DETR_ROOT, ACT_ENV_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from policy import ACTPolicy
import gym_pusht  # registers gym_pusht/PushT-v0

def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('OFFICIAL_ROOT:', OFFICIAL_ROOT)
print('OFFICIAL_DETR_ROOT:', OFFICIAL_DETR_ROOT)
print('ACT_ENV_ROOT:', ACT_ENV_ROOT)
print('Device:', device)

## 2. Resolve PushT Dataset
Locate the Zarr dataset and map it into arrays for images, state, actions, and episode boundaries.

In [ ]:
def resolve_pusht_dataset_path() -> Path:
    for p in DATA_CANDIDATES:
        if p.is_dir():
            try:
                root = zarr.open(str(p), mode='r')
                if 'data' in root and 'meta' in root:
                    return p
            except Exception:
                pass
        if p.is_file() and p.suffix == '.zip':
            try:
                root = zarr.open(str(p), mode='r')
                if 'data' in root and 'meta' in root:
                    return p
            except Exception:
                pass
    raise FileNotFoundError('Could not resolve PushT dataset from ACT_pusht_task/data')

DATASET_PATH = resolve_pusht_dataset_path()
ROOT = zarr.open(str(DATASET_PATH), mode='r')

raw_img = ROOT['data']['img']
raw_state = np.asarray(ROOT['data']['state'][:, :5], dtype=np.float32)
raw_action = np.asarray(ROOT['data']['action'][:], dtype=np.float32)
episode_ends = np.asarray(ROOT['meta']['episode_ends'][:], dtype=np.int64)

# PushT state(5) -> ACT qpos(14) with zero padding.
state_6 = np.concatenate([
    raw_state[:, :4],
    np.sin(raw_state[:, 4:5]),
    np.cos(raw_state[:, 4:5]),
], axis=1).astype(np.float32)
qpos_14 = np.zeros((state_6.shape[0], 14), dtype=np.float32)
qpos_14[:, :6] = state_6

# PushT action(2) -> ACT action(14) with zero padding.
action_14 = np.zeros((raw_action.shape[0], 14), dtype=np.float32)
action_14[:, :2] = raw_action

print('Resolved dataset:', DATASET_PATH)
print('Images shape:', raw_img.shape)
print('Qpos shape:', qpos_14.shape)
print('Action shape:', action_14.shape)
print('Episodes:', len(episode_ends))

## 3. Dataset Adapter and Splits
Convert PushT arrays to the ACT input format and create train/val/test splits.

In [ ]:
def make_episode_ranges(episode_ends_arr):
    starts = np.concatenate([[0], episode_ends_arr[:-1]])
    return list(zip(starts.tolist(), episode_ends_arr.tolist()))

def build_norm_stats(qpos_arr, action_arr):
    qpos_mean = qpos_arr.mean(axis=0)
    qpos_std = qpos_arr.std(axis=0)
    qpos_std = np.clip(qpos_std, 1e-3, np.inf)

    action_mean = action_arr.mean(axis=0)
    action_std = action_arr.std(axis=0)
    action_std = np.clip(action_std, 1e-3, np.inf)

    return {
        'qpos_mean': qpos_mean.astype(np.float32),
        'qpos_std': qpos_std.astype(np.float32),
        'action_mean': action_mean.astype(np.float32),
        'action_std': action_std.astype(np.float32),
    }

class PushTOfficialACTDataset(Dataset):
    def __init__(self, episode_ids, episode_ranges, image_arr, qpos_arr, action_arr, norm_stats, chunk_size=16):
        self.episode_ids = list(episode_ids)
        self.episode_ranges = episode_ranges
        self.image_arr = image_arr
        self.qpos_arr = qpos_arr
        self.action_arr = action_arr
        self.norm_stats = norm_stats
        self.chunk_size = int(chunk_size)

    def __len__(self):
        return len(self.episode_ids)

    def __getitem__(self, idx):
        ep_id = self.episode_ids[idx]
        ep_start, ep_end = self.episode_ranges[ep_id]
        ep_len = ep_end - ep_start

        start_ts = np.random.randint(0, ep_len)
        global_idx = ep_start + start_ts

        qpos = self.qpos_arr[global_idx].copy()

        action_seq = np.zeros((self.chunk_size, 14), dtype=np.float32)
        is_pad = np.ones((self.chunk_size,), dtype=bool)

        avail = min(self.chunk_size, ep_end - global_idx)
        action_seq[:avail] = self.action_arr[global_idx:global_idx + avail]
        is_pad[:avail] = False

        image = np.asarray(self.image_arr[global_idx], dtype=np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))
        image = np.expand_dims(image, axis=0)

        qpos = (qpos - self.norm_stats['qpos_mean']) / self.norm_stats['qpos_std']
        action_seq = (action_seq - self.norm_stats['action_mean']) / self.norm_stats['action_std']

        return (
            torch.from_numpy(image).float(),
            torch.from_numpy(qpos).float(),
            torch.from_numpy(action_seq).float(),
            torch.from_numpy(is_pad),
        )

episode_ranges = make_episode_ranges(episode_ends)
num_episodes = len(episode_ranges)
all_episode_ids = np.arange(num_episodes)
np.random.shuffle(all_episode_ids)

# Proper 80/10/10 split at episode level: train / val / test.
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-8

n_train = int(TRAIN_RATIO * num_episodes)
n_val = int(VAL_RATIO * num_episodes)
train_ids = all_episode_ids[:n_train]
val_ids = all_episode_ids[n_train:n_train + n_val]
test_ids = all_episode_ids[n_train + n_val:]

norm_stats = build_norm_stats(qpos_14, action_14)

CHUNK_SIZE = 16
BATCH_SIZE = 32

train_ds = PushTOfficialACTDataset(train_ids, episode_ranges, raw_img, qpos_14, action_14, norm_stats, chunk_size=CHUNK_SIZE)
val_ds = PushTOfficialACTDataset(val_ids, episode_ranges, raw_img, qpos_14, action_14, norm_stats, chunk_size=CHUNK_SIZE)
test_ds = PushTOfficialACTDataset(test_ids, episode_ranges, raw_img, qpos_14, action_14, norm_stats, chunk_size=CHUNK_SIZE)

num_workers = 0
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available())
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())

sample = next(iter(train_dl))
print('Train episodes:', len(train_ds), '| Val episodes:', len(val_ds), '| Test episodes:', len(test_ds))
print('image:', sample[0].shape, 'qpos:', sample[1].shape, 'action:', sample[2].shape, 'is_pad:', sample[3].shape)

## 4. Policy Configuration
Build the ACT policy using the official implementation and set training hyperparameters.

In [ ]:
def build_official_act_policy_for_notebook(chunk_size=16, lr=3e-5, kl_weight=10):
    # official_act_repo_probe/detr/main.py parser has required CLI args.
    argv_backup = sys.argv[:]
    sys.argv = [
        argv_backup[0],
        '--ckpt_dir', 'tmp',
        '--policy_class', 'ACT',
        '--task_name', 'pusht_dataset_only',
        '--seed', '0',
        '--num_epochs', '1',
        '--batch_size', '2',
        '--lr', str(lr),
    ]
    try:
        args_override = {
            'lr': lr,
            'num_queries': chunk_size,
            'kl_weight': kl_weight,
            'hidden_dim': 512,
            'dim_feedforward': 3200,
            'lr_backbone': 1e-5,
            'backbone': 'resnet18',
            'enc_layers': 4,
            'dec_layers': 7,
            'nheads': 8,
            'camera_names': ['main'],
        }
        policy = ACTPolicy(args_override)
        return policy
    finally:
        sys.argv = argv_backup

policy = build_official_act_policy_for_notebook(chunk_size=CHUNK_SIZE, lr=3e-5, kl_weight=10)
policy.to(device)
optimizer = policy.configure_optimizers()
print('Policy built from official_act_repo_probe')
print('Parameters:', sum(p.numel() for p in policy.parameters()))

## 5. Training and Rollout Evaluation
Train the policy, log metrics, and select the best checkpoint based on simulator rollouts.

In [ ]:
import time

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'official_fresh'
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
METRICS_DIR = OUTPUT_DIR / 'metrics'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT_PATH = CKPT_DIR / 'policy_best_rollout.ckpt'
LAST_CKPT_PATH = CKPT_DIR / 'policy_last.ckpt'
CSV_PATH = METRICS_DIR / 'epoch_metrics.csv'

RUN_TRAINING = True
# Set to True only for quick debugging; False runs real training.
SMOKE_TEST = True
NUM_EPOCHS = 2 if SMOKE_TEST else 300

# If True, always try to load PRETRAINED_CKPT_PATH.
INIT_FROM_PRETRAINED = False
# If True, auto-resume only when checkpoint quality is above threshold.
AUTO_RESUME_IF_GOOD = True
MIN_RESUME_COVERAGE = 0.03
PRETRAINED_CKPT_PATH = OUTPUT_DIR / 'checkpoints' / 'policy_best_rollout.ckpt'
LOAD_STRICT = True

# True simulator rollout config (blue-ball pushes T onto target T).
SIM_ROLLOUT_EPISODES = 2 if SMOKE_TEST else 10
SIM_ROLLOUT_MAX_STEPS = 200
# Official ACT temporal-agg uses a small decay constant (~0.01).
SIM_ROLLOUT_TEMPORAL_BETA = 0.01

# Quick beta sanity sweep before long training.
RUN_BETA_SWEEP = False if SMOKE_TEST else True
BETA_SWEEP_VALUES = [0.005, 0.01, 0.02, 0.05]
BETA_SWEEP_EPISODES = 2 if SMOKE_TEST else 6

resume_applied = False
if PRETRAINED_CKPT_PATH.is_file():
    ckpt = torch.load(PRETRAINED_CKPT_PATH, map_location=device)
    state = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    sel = ckpt.get('selection_metric', {}) if isinstance(ckpt, dict) else {}
    ckpt_cov = float(sel.get('rollout_mean_final_coverage', 0.0))

    should_resume = INIT_FROM_PRETRAINED or (
        AUTO_RESUME_IF_GOOD and ckpt_cov >= MIN_RESUME_COVERAGE
    )
    if should_resume:
        try:
            missing, unexpected = policy.load_state_dict(state, strict=LOAD_STRICT)
            resume_applied = True
            print('Loaded pretrained from:', PRETRAINED_CKPT_PATH)
            print('Loaded ckpt coverage:', ckpt_cov)
            print('Missing keys:', len(missing), 'Unexpected keys:', len(unexpected))
        except RuntimeError as e:
            # Common when architecture changed (e.g., hidden_dim 256 -> 512).
            print('Pretrained load skipped due to state_dict mismatch.')
            print('Reason:', str(e).split('\n')[0])
            print('Training will start from current initialization.')
    else:
        print('Checkpoint found but skipped (coverage below threshold).')
        print('Found coverage:', ckpt_cov, '| required >=', MIN_RESUME_COVERAGE)
else:
    print('No pretrained checkpoint found. Starting from scratch.')

def run_epoch(data_loader, train_mode: bool):
    if train_mode:
        policy.train()
    else:
        policy.eval()

    losses = []
    l1_first2_vals = []

    for image_data, qpos_data, action_data, is_pad in data_loader:
        image_data = image_data.to(device, non_blocking=True)
        qpos_data = qpos_data.to(device, non_blocking=True)
        action_data = action_data.to(device, non_blocking=True)
        is_pad = is_pad.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            out = policy(qpos_data, image_data, action_data, is_pad)
            loss = out['loss']

            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                optimizer.step()

        losses.append(float(loss.detach().item()))

        # Use eval mode for metrics so dropout/BatchNorm do not perturb logging.
        was_training = policy.training
        policy.eval()
        with torch.no_grad():
            pred_actions = policy(qpos_data, image_data)
        if was_training:
            policy.train()
        gt = action_data[:, :pred_actions.shape[1], :2]
        pr = pred_actions[:, :, :2]
        mask = (~is_pad[:, :pred_actions.shape[1]]).unsqueeze(-1)
        abs_err = torch.abs(pr - gt) * mask
        denom = mask.sum().clamp_min(1)
        l1_first2 = abs_err.sum() / denom
        l1_first2_vals.append(float(l1_first2.item()))

    return float(np.mean(losses)), float(np.mean(l1_first2_vals))

def _sim_state_from_obs_info(obs, info):
    # Build 14-dim qpos expected by official ACT policy.
    agent_pos = np.asarray(obs['agent_pos'], dtype=np.float32)
    block_pose = np.asarray(info['block_pose'], dtype=np.float32)
    state_6 = np.array([
        agent_pos[0],
        agent_pos[1],
        block_pose[0],
        block_pose[1],
        np.sin(block_pose[2]),
        np.cos(block_pose[2]),
    ], dtype=np.float32)
    qpos = np.zeros((14,), dtype=np.float32)
    qpos[:6] = state_6
    qpos = (qpos - norm_stats['qpos_mean']) / norm_stats['qpos_std']
    return qpos

def _sim_img_from_obs(obs):
    # Obs pixels are (H, W, C) uint8 -> (1, C, H, W) float in [0,1].
    img = np.asarray(obs['pixels'], dtype=np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    return img

def _denorm_action14(pred_action14):
    return pred_action14 * norm_stats['action_std'] + norm_stats['action_mean']

def rollout_episode_sim(seed=None, max_steps=200, temporal_agg_beta=0.01, video_path=None):
    env = gym.make(
        'gym_pusht/PushT-v0',
        obs_type='pixels_agent_pos',
        render_mode='rgb_array',
        workspace_size=512,
        success_threshold=0.85,
        visualization_width=672,
        visualization_height=672,
    )

    obs, info = env.reset(seed=seed)
    frames = []
    rewards = []
    coverage = 0.0
    success = False

    predicted_chunks = []
    chunk_start_steps = []

    policy.eval()
    for t in range(max_steps):
        img = _sim_img_from_obs(obs)
        qpos = _sim_state_from_obs_info(obs, info)

        img_t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(device=device, dtype=torch.float32)
        qpos_t = torch.from_numpy(qpos).unsqueeze(0).to(device=device, dtype=torch.float32)

        with torch.no_grad():
            chunk_pred = policy(qpos_t, img_t)[0].detach().cpu().numpy()  # (num_queries, 14)

        chunk_denorm = _denorm_action14(chunk_pred)
        predicted_chunks.append(chunk_denorm)
        chunk_start_steps.append(t)

        # Temporal aggregation over available predicted actions.
        candidates = []
        ages = []
        for ch, st in zip(predicted_chunks, chunk_start_steps):
            offset = t - st
            if 0 <= offset < len(ch):
                candidates.append(ch[offset])
                ages.append(t - st)

        if candidates:
            candidates = np.stack(candidates, axis=0)
            w = np.exp(-temporal_agg_beta * np.asarray(ages, dtype=np.float32))
            w = w / np.sum(w)
            action14 = np.sum(candidates * w[:, None], axis=0).astype(np.float32)
        else:
            action14 = chunk_denorm[0].astype(np.float32)

        # PushT expects 2D action in pixel workspace coordinates.
        action2 = np.nan_to_num(action14[:2], nan=256.0, posinf=511.0, neginf=0.0)
        action2 = np.clip(action2, 0.0, 511.0).astype(np.float32)
        obs, reward, terminated, truncated, info = env.step(action2)

        rewards.append(float(reward))
        coverage = float(info.get('coverage', coverage))
        success = bool(info.get('is_success', False))
        if video_path is not None:
            frames.append(env.render())

        if terminated or truncated:
            break

    env.close()

    if video_path is not None and frames:
        import imageio.v2 as imageio
        Path(video_path).parent.mkdir(parents=True, exist_ok=True)
        imageio.mimsave(str(video_path), frames, fps=10, macro_block_size=None)

    return {
        'episode_return': float(np.sum(rewards)) if rewards else 0.0,
        'episode_length': len(rewards),
        'max_reward': float(np.max(rewards)) if rewards else 0.0,
        'final_coverage': coverage,
        'success': success,
        'frames': frames,
    }

def evaluate_rollout_metrics(num_episodes=5, max_steps=200, temporal_agg_beta=0.01, seed=0):
    episodes = []
    for ep in range(num_episodes):
        episodes.append(
            rollout_episode_sim(
                seed=seed + ep,
                max_steps=max_steps,
                temporal_agg_beta=temporal_agg_beta,
                video_path=None,
            )
        )

    return {
        'rollout_mean_return': float(np.mean([e['episode_return'] for e in episodes])) if episodes else 0.0,
        'rollout_mean_max_reward': float(np.mean([e['max_reward'] for e in episodes])) if episodes else 0.0,
        'rollout_mean_final_coverage': float(np.mean([e['final_coverage'] for e in episodes])) if episodes else 0.0,
        'rollout_success_rate': float(np.mean([e['success'] for e in episodes])) if episodes else 0.0,
    }

if RUN_BETA_SWEEP:
    print('Running beta sweep before training...')
    best_beta = SIM_ROLLOUT_TEMPORAL_BETA
    best_cov = -float('inf')
    for beta in BETA_SWEEP_VALUES:
        m = evaluate_rollout_metrics(
            num_episodes=BETA_SWEEP_EPISODES,
            max_steps=SIM_ROLLOUT_MAX_STEPS,
            temporal_agg_beta=beta,
            seed=4242,
        )
        cov = m['rollout_mean_final_coverage']
        succ = m['rollout_success_rate']
        print(f'Beta {beta:.3f} -> coverage={cov:.6f}, success={succ:.6f}')
        if cov > best_cov:
            best_cov = cov
            best_beta = beta
    SIM_ROLLOUT_TEMPORAL_BETA = float(best_beta)
    print('Selected SIM_ROLLOUT_TEMPORAL_BETA =', SIM_ROLLOUT_TEMPORAL_BETA)

if RUN_TRAINING:
    best_rollout_success = -float('inf')
    best_rollout_max_reward = -float('inf')
    best_rollout_coverage = -float('inf')

    fieldnames = [
        'epoch',
        'train_loss',
        'train_l1_action2',
        'val_loss',
        'val_l1_action2',
        'test_loss',
        'test_l1_action2',
        'rollout_mean_return',
        'rollout_mean_max_reward',
        'rollout_mean_final_coverage',
        'rollout_success_rate',
        'best_rollout_success_so_far',
        'best_rollout_max_reward_so_far',
        'best_rollout_coverage_so_far',
    ]
    with open(CSV_PATH, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    run_t0 = time.perf_counter()
    for epoch in tqdm(range(NUM_EPOCHS), desc='Official-based PushT ACT training'):
        epoch_t0 = time.perf_counter()

        train_loss, train_l1 = run_epoch(train_dl, train_mode=True)
        val_loss, val_l1 = run_epoch(val_dl, train_mode=False)
        test_loss, test_l1 = run_epoch(test_dl, train_mode=False)

        rollout_metrics = evaluate_rollout_metrics(
            num_episodes=SIM_ROLLOUT_EPISODES,
            max_steps=SIM_ROLLOUT_MAX_STEPS,
            temporal_agg_beta=SIM_ROLLOUT_TEMPORAL_BETA,
            seed=1000 + epoch * 50,
        )
        rollout_return = rollout_metrics['rollout_mean_return']
        rollout_max_reward = rollout_metrics['rollout_mean_max_reward']
        rollout_coverage = rollout_metrics['rollout_mean_final_coverage']
        rollout_success = rollout_metrics['rollout_success_rate']

        improved = (
            (rollout_success > best_rollout_success)
            or (rollout_success == best_rollout_success and rollout_max_reward > best_rollout_max_reward)
            or (
                rollout_success == best_rollout_success
                and rollout_max_reward == best_rollout_max_reward
                and rollout_coverage > best_rollout_coverage
            )
        )
        if improved:
            best_rollout_success = rollout_success
            best_rollout_max_reward = rollout_max_reward
            best_rollout_coverage = rollout_coverage
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': policy.state_dict(),
                'norm_stats': norm_stats,
                'selection_metric': {
                    'rollout_success_rate': best_rollout_success,
                    'rollout_mean_max_reward': best_rollout_max_reward,
                    'rollout_mean_final_coverage': best_rollout_coverage,
                },
                'config': {
                    'chunk_size': CHUNK_SIZE,
                    'batch_size': BATCH_SIZE,
                    'sim_rollout_episodes': SIM_ROLLOUT_EPISODES,
                    'sim_rollout_max_steps': SIM_ROLLOUT_MAX_STEPS,
                    'sim_rollout_temporal_beta': SIM_ROLLOUT_TEMPORAL_BETA,
                    'source': 'official_act_repo_probe + PushT dataset + true sim rollout',
                },
            }, BEST_CKPT_PATH)

        with open(CSV_PATH, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writerow({
                'epoch': epoch + 1,
                'train_loss': train_loss,
                'train_l1_action2': train_l1,
                'val_loss': val_loss,
                'val_l1_action2': val_l1,
                'test_loss': test_loss,
                'test_l1_action2': test_l1,
                'rollout_mean_return': rollout_return,
                'rollout_mean_max_reward': rollout_max_reward,
                'rollout_mean_final_coverage': rollout_coverage,
                'rollout_success_rate': rollout_success,
                'best_rollout_success_so_far': best_rollout_success,
                'best_rollout_max_reward_so_far': best_rollout_max_reward,
                'best_rollout_coverage_so_far': best_rollout_coverage,
            })

        epoch_dt = time.perf_counter() - epoch_t0
        avg_epoch = (time.perf_counter() - run_t0) / float(epoch + 1)
        eta_min = avg_epoch * float(NUM_EPOCHS - (epoch + 1)) / 60.0

        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
            f"train_loss={train_loss:.6f} | train_l1_action2={train_l1:.6f} | "
            f"val_loss={val_loss:.6f} | val_l1_action2={val_l1:.6f} | "
            f"test_loss={test_loss:.6f} | test_l1_action2={test_l1:.6f}"
        )
        print(
            f"Rollout metrics -> "
            f"mean_return={rollout_return:.6f} | "
            f"mean_max_reward={rollout_max_reward:.6f} | "
            f"mean_final_coverage={rollout_coverage:.6f} | "
            f"success_rate={rollout_success:.6f}"
        )
        print(
            f"Best rollout so far -> "
            f"success={best_rollout_success:.6f} | "
            f"max_reward={best_rollout_max_reward:.6f} | "
            f"coverage={best_rollout_coverage:.6f} | "
            f"ckpt={BEST_CKPT_PATH}"
        )
        print(
            f"Timing -> epoch_sec={epoch_dt:.2f} | avg_epoch_sec={avg_epoch:.2f} | eta_min={eta_min:.1f}"
        )

    torch.save({'model_state_dict': policy.state_dict(), 'norm_stats': norm_stats}, LAST_CKPT_PATH)
    print('Best checkpoint:', BEST_CKPT_PATH)
    print('Last checkpoint:', LAST_CKPT_PATH)
    print('Metrics CSV:', CSV_PATH)
else:
    print('RUN_TRAINING=False; training skipped.')

## 6. Final Evaluation and Visualization
Run final rollouts, summarize metrics, and export a sample video with a quick action-curve diagnostic.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Video, display

if BEST_CKPT_PATH.is_file():
    ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
    state = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    policy.load_state_dict(state, strict=True)
    policy.eval()

# 1) Final true simulator rollout summary.
final_rollout_summary = evaluate_rollout_metrics(
    num_episodes=5,
    max_steps=SIM_ROLLOUT_MAX_STEPS,
    temporal_agg_beta=SIM_ROLLOUT_TEMPORAL_BETA,
    seed=2026,
)
print('Final simulator-rollout summary:')
print('rollout_mean_return:', final_rollout_summary['rollout_mean_return'])
print('rollout_mean_max_reward:', final_rollout_summary['rollout_mean_max_reward'])
print('rollout_mean_final_coverage:', final_rollout_summary['rollout_mean_final_coverage'])
print('rollout_success_rate:', final_rollout_summary['rollout_success_rate'])

# 2) Final offline metrics on held-out test loader.
def evaluate_offline_metrics_on_loader(data_loader):
    policy.eval()
    losses = []
    l1_first2_vals = []
    with torch.no_grad():
        for image_data, qpos_data, action_data, is_pad in data_loader:
            image_data = image_data.to(device, non_blocking=True)
            qpos_data = qpos_data.to(device, non_blocking=True)
            action_data = action_data.to(device, non_blocking=True)
            is_pad = is_pad.to(device, non_blocking=True)

            out = policy(qpos_data, image_data, action_data, is_pad)
            losses.append(float(out['loss'].detach().item()))

            pred_actions = policy(qpos_data, image_data)
            gt = action_data[:, :pred_actions.shape[1], :2]
            pr = pred_actions[:, :, :2]
            mask = (~is_pad[:, :pred_actions.shape[1]]).unsqueeze(-1)
            abs_err = torch.abs(pr - gt) * mask
            denom = mask.sum().clamp_min(1)
            l1_first2 = abs_err.sum() / denom
            l1_first2_vals.append(float(l1_first2.item()))

    return {
        'test_loss': float(np.mean(losses)) if losses else 0.0,
        'test_l1_action2': float(np.mean(l1_first2_vals)) if l1_first2_vals else 0.0,
    }

test_metrics = evaluate_offline_metrics_on_loader(test_dl)
print('Final offline test metrics:')
print('test_loss:', test_metrics['test_loss'])
print('test_l1_action2:', test_metrics['test_l1_action2'])

# 3) Create and display the actual PushT simulator video.
video_dir = OUTPUT_DIR / 'videos'
video_dir.mkdir(parents=True, exist_ok=True)
video_path = video_dir / 'sim_episode_00.mp4'

episode_video = rollout_episode_sim(
    seed=3030,
    max_steps=SIM_ROLLOUT_MAX_STEPS,
    temporal_agg_beta=SIM_ROLLOUT_TEMPORAL_BETA,
    video_path=str(video_path),
)

print('Video rollout max reward:', episode_video['max_reward'])
print('Video rollout final coverage:', episode_video['final_coverage'])
print('Video rollout success:', episode_video['success'])
print('Video saved to:', video_path)

if video_path.is_file():
    display(Video(str(video_path), embed=True))

# 4) Keep one static action-curve diagnostic from test batch.
test_batch = next(iter(test_dl))
image_data, qpos_data, action_data, is_pad = [x.to(device) for x in test_batch]
with torch.no_grad():
    pred_actions = policy(qpos_data, image_data)

T = pred_actions.shape[1]
valid_lens = (~is_pad[:, :T]).sum(dim=1)
b = int(torch.argmax(valid_lens).item())
valid_len = int(valid_lens[b].item())

if valid_len <= 0:
    print('No valid timesteps found in this test batch for curve plotting.')
else:
    a_mean = torch.as_tensor(norm_stats['action_mean'], device=device, dtype=pred_actions.dtype)
    a_std = torch.as_tensor(norm_stats['action_std'], device=device, dtype=pred_actions.dtype)

    gt_xy = (action_data[b, :valid_len, :2] * a_std[:2] + a_mean[:2]).detach().cpu().numpy()
    pr_xy = (pred_actions[b, :valid_len, :2] * a_std[:2] + a_mean[:2]).detach().cpu().numpy()
    t_idx = np.arange(valid_len)

    plt.figure(figsize=(8, 4))
    line_style = '-o' if valid_len <= 3 else '-'
    plt.plot(t_idx, gt_xy[:, 0], line_style, label='gt_action_x')
    plt.plot(t_idx, pr_xy[:, 0], line_style, label='pred_action_x')
    plt.plot(t_idx, gt_xy[:, 1], line_style, label='gt_action_y')
    plt.plot(t_idx, pr_xy[:, 1], line_style, label='pred_action_y')
    plt.title(f'Test sample (max valid len={valid_len})')
    plt.xlabel('Timestep in chunk')
    plt.ylabel('Action value (denormalized)')
    plt.xticks(t_idx if valid_len <= 12 else None)
    plt.legend()
    plt.tight_layout()
    plt.show()

print('Done. Final simulator summary + offline test metrics + simulator video + test curve generated.')